# INF2006 G12 - Stage the raw corpus: S3 → HDFS

Copies the 5 corpus sources from S3 into HDFS **as original bytes** (nothing unzipped), replication 2,
and proves each file arrived intact (sha256 of the S3 download = sha256 read back from HDFS).

| source | S3 | HDFS (`/corpus/raw/…`) | files |
|---|---|---|---|
| prices | `PRICE/` | `prices/v1_2013-2017/acq_2026-10-28/` | 1 |
| sec_inventory | `SEC_Filing_Inventory.csv` | `sec_inventory/v1/acq_2026-10-28/` | 1 |
| sec_edgar | `SEC EDGAR/` | `sec_edgar/v1/acq_2026-10-28/` | 311 |
| wikinews | `Wikinews/` | `wikinews/20261001_meta-history/acq_2026-10-29/` | 2 |
| gdelt | `GDELT/` (2015-07-01 to 2017-06-30) | `gdelt/v1_events_2015-2017/acq_2026-10-29/` | 731 |

Manifests (staging record + summary) go to `/corpus/raw/_manifests/<source>/<version>/acq_<date>/`.
The `acq_` dates are the original download dates, so everyone's HDFS paths are identical.

**How to run**
1. SSH to the EMR primary node, then `tmux new -s stage` (if SSH drops: `tmux attach -t stage`).
2. Paste the cells below in order. Run everything in the same shell (your tmux session).
   If you ever end up in a different shell, paste cells 1 and 3 again before continuing.
3. Raw folders are immutable: a staging cell stops if its HDFS folder already exists.

## 1. Config
Change `B` only if your copy of the data is in another bucket.

In [ ]:
export B=inf2006-bigdata
export STAGE=/mnt/g12/stage
sudo mkdir -p /mnt/g12 && sudo chown hadoop:hadoop /mnt/g12 && mkdir -p "$STAGE"

# source | S3 URI | version | acq date | expected files | mode (prefix = folder, single = one object)
SOURCES=(
  "prices|s3://$B/PRICE/|v1_2013-2017|2026-10-28|1|prefix"
  "sec_inventory|s3://$B/SEC_Filing_Inventory.csv|v1|2026-10-28|1|single"
  "sec_edgar|s3://$B/SEC EDGAR/|v1|2026-10-28|311|prefix"
  "wikinews|s3://$B/Wikinews/|20261001_meta-history|2026-10-29|2|prefix"
  "gdelt|s3://$B/GDELT/|v1_events_2015-2017|2026-10-29|731|prefix"
)

# Sources you plan to stage. Delete the ones you don't need (e.g. gdelt) for development.
STAGE_SOURCES="prices sec_inventory sec_edgar wikinews gdelt"
want_src() { [[ " $STAGE_SOURCES " == *" $1 "* ]]; }

raw_path() { echo "/corpus/raw/$1/$2/acq_$3"; }             # <src> <version> <acq>
man_path() { echo "/corpus/raw/_manifests/$1/$2/acq_$3"; }
# "<files> <bytes>" under an S3 URI (folder markers ending in / are ignored)
s3_stats() { aws s3 ls "$1" --recursive | awk 'substr($0,length($0))!="/" {n++; b+=$3} END {printf "%d %.0f\n", n, b}'; }
src_row()  { local r; for r in "${SOURCES[@]}"; do [[ ${r%%|*} == "$1" ]] && { echo "$r"; return 0; }; done
             echo "unknown source: $1" >&2; return 1; }
echo "config loaded: bucket=$B stage=$STAGE sources=${#SOURCES[@]} staging=[$STAGE_SOURCES]"

## 2. Preflight (read-only)
All lines should say PASS. Sources not in `STAGE_SOURCES` are marked `(skip)`.
The disk check needs room on `/mnt` for the largest selected source (GDELT, if selected, is about 9-10 GB).

In [ ]:
echo "== S3 objects vs expected =="
printf '%-14s %8s %8s %14s  %s\n' source expected found bytes result
max=0
for r in "${SOURCES[@]}"; do
  IFS='|' read -r src s3 ver acq want mode <<< "$r"
  read -r n b <<< "$(s3_stats "$s3")"
  [[ $n -eq $want ]] && res=PASS || res=FAIL
  if want_src "$src"; then (( b > max )) && max=$b; else res="$res (skip)"; fi
  printf '%-14s %8s %8s %14s  %s\n' "$src" "$want" "$n" "$b" "$res"
done

echo; echo "== Local disk on $STAGE =="
df -h "$STAGE"
free=$(df -B1 --output=avail "$STAGE" | tail -1)
need=$(( max + 1073741824 ))
if (( free >= need )); then echo "PASS: $((free/1024**3)) GiB free >= $((need/1024**3)) GiB needed (largest selected source + 1 GiB)"
else echo "FAIL: $((free/1024**3)) GiB free < $((need/1024**3)) GiB needed. Free space on /mnt or use a bigger primary-node volume."; fi

echo; echo "== HDFS =="
hdfs dfs -ls / > /dev/null && echo "PASS: HDFS reachable" || echo "FAIL: HDFS not reachable"
echo "cluster dfs.replication = $(hdfs getconf -confKey dfs.replication)  (every write below forces 2)"
hdfs dfs -df -h /

echo; echo "== Raw targets (must not exist yet) =="
for r in "${SOURCES[@]}"; do
  IFS='|' read -r src s3 ver acq want mode <<< "$r"
  p=$(raw_path "$src" "$ver" "$acq")
  hdfs dfs -test -e "$p" && echo "EXISTS  $p  (its cell will STOP)" || echo "PASS    $p"
done

## 3. Define `stage_source`
For one source: stop if the HDFS folder exists → check disk → download from S3 to `/mnt` → local sha256 →
`hdfs dfs -put` (replication 2) → read every file back from HDFS and sha256 it (4 in parallel) →
write `staging_record.csv` + `staging_summary.txt` to `_manifests` → delete the local copy.
Runs in a subshell, so an error never closes your SSH session.

In [ ]:
stage_source() (
  set -o pipefail
  row=$(src_row "$1") || exit 2
  IFS='|' read -r src s3 ver acq want mode <<< "$row"
  raw=$(raw_path "$src" "$ver" "$acq"); man=$(man_path "$src" "$ver" "$acq")
  work=$STAGE/$src; data=$work/data; rec=$work/staging_record.csv; summ=$work/staging_summary.txt
  echo "== $src: $s3 -> $raw"

  # 0. raw is immutable
  if hdfs dfs -test -e "$raw"; then echo "STOP: $raw already exists (raw is immutable). See the reset cell."; exit 3; fi

  # 1. check S3 count + local disk, then download
  read -r n_s3 b_s3 <<< "$(s3_stats "$s3")"
  [[ $n_s3 -eq $want ]] || { echo "STOP: S3 has $n_s3 files, expected $want"; exit 4; }
  rm -rf "$work"; mkdir -p "$data"
  free=$(df -B1 --output=avail "$STAGE" | tail -1)
  (( free > b_s3 + 1073741824 )) || { echo "STOP: need $((b_s3/1024**2)) MiB + 1 GiB on $STAGE, only $((free/1024**2)) MiB free"; exit 5; }
  echo "[1/5] download $n_s3 files, $((b_s3/1024**2)) MiB"
  t0=$(date +%s)
  if [[ $mode == single ]]; then aws s3 cp "$s3" "$data/" --only-show-errors
  else aws s3 cp "$s3" "$data/" --recursive --only-show-errors; fi || { echo "STOP: S3 download failed"; exit 1; }
  (cd "$data" && find . -type f -printf '%P\n' | LC_ALL=C sort) > "$work/files.txt"
  n=$(wc -l < "$work/files.txt")
  [[ $n -eq $want ]] || { echo "STOP: downloaded $n files, expected $want"; exit 4; }

  # 2. local sha256  ->  file|bytes|sha256
  echo "[2/5] local sha256"
  while IFS= read -r f; do
    printf '%s|%s|%s\n' "$f" "$(stat -c %s "$data/$f")" "$(sha256sum < "$data/$f" | cut -d' ' -f1)"
  done < "$work/files.txt" > "$work/local.txt"

  # 3. put into HDFS (one bulk put, replication 2)
  echo "[3/5] hdfs put"
  t1=$(date +%s)
  hdfs dfs -mkdir -p "$(dirname "$raw")" || exit 1
  hdfs dfs -D dfs.replication=2 -put "$data" "$raw" || { echo "STOP: put failed; a partial $raw may exist (see the reset cell)"; exit 1; }

  # 4. read back from HDFS: sha256 per file, bytes + replication from one listing
  echo "[4/5] HDFS read-back sha256"
  t2=$(date +%s)
  RAW=$raw xargs -a "$work/files.txt" -d '\n' -P 4 -I{} \
    bash -c 'printf "%s|%s\n" "$1" "$(hdfs dfs -cat "$RAW/$1" | sha256sum | cut -d" " -f1)"' _ {} > "$work/hdfs_sha.txt"
  hdfs dfs -ls -R "$raw" | awk -v p="$raw/" '$1 !~ /^d/ && NF >= 8 {print substr($0, index($0, p) + length(p)) "|" $5 "|" $2}' > "$work/hdfs_meta.txt"
  t3=$(date +%s)

  # 5. staging record + summary -> _manifests
  echo "[5/5] record"
  echo "staged_at_utc,host,s3_uri,hdfs_path,local_bytes,hdfs_bytes,local_sha256,hdfs_sha256,match" > "$rec"
  awk -F'|' -v OFS=, -v now="$(date -u +%FT%TZ)" -v host="$(hostname)" -v s3="$s3" -v mode="$mode" -v raw="$raw" '
    FILENAME == ARGV[1] { hs[$1] = $2; next }
    FILENAME == ARGV[2] { hb[$1] = $2; next }
    { uri = (mode == "single") ? s3 : s3 $1
      ok  = ($2 == hb[$1] && $3 == hs[$1]) ? "yes" : "NO"
      print now, host, uri, raw "/" $1, $2, hb[$1], $3, hs[$1], ok }' \
    "$work/hdfs_sha.txt" "$work/hdfs_meta.txt" "$work/local.txt" >> "$rec"
  bad=$(grep -c ',NO$' "$rec")
  n_hdfs=$(wc -l < "$work/hdfs_meta.txt")
  rep_bad=$(awk -F'|' '$3 != 2' "$work/hdfs_meta.txt" | wc -l)
  { echo "source=$src version=$ver acq=$acq s3=$s3"
    echo "hdfs=$raw"
    echo "files_expected=$want files_local=$n files_hdfs=$n_hdfs mismatches=$bad replication_not_2=$rep_bad"
    echo "download_s=$((t1-t0)) put_s=$((t2-t1)) readback_s=$((t3-t2))"
    echo "hdfs_du(bytes, bytes_with_replicas, path)=$(hdfs dfs -du -s "$raw")"
  } | tee "$summ"
  hdfs dfs -mkdir -p "$man" && hdfs dfs -D dfs.replication=2 -put -f "$rec" "$summ" "$man/"
  rm -rf "$data"

  if [[ $bad -eq 0 && $rep_bad -eq 0 && $n_hdfs -eq $want ]]; then echo "OK: $src staged and verified -> $man"
  else echo "MISMATCH: see $rec"; exit 6; fi
)
echo "stage_source defined"

## 4. Stage each source
One cell per source. Each must end with `OK:`.
Skip the cells for sources not in `STAGE_SOURCES`.

### 4a. Prices (1 file, ~28 MB, < 1 min)

In [ ]:
stage_source prices

### 4b. SEC filing inventory (1 file, ~51 KB)

In [ ]:
stage_source sec_inventory

### 4c. SEC EDGAR filings (311 files, ~2.8 GB, a few minutes)

In [ ]:
stage_source sec_edgar

### 4d. Wikinews (2 `.bz2` files, ~630 MB)

In [ ]:
stage_source wikinews

### 4e. GDELT (731 `.zip` files, ~9-10 GB)
Largest source, and the peak for `/mnt` disk use. Expect roughly 15-30 min. Keep tmux open.

In [ ]:
stage_source gdelt

## 5. Final check
Every selected source should say PASS; unselected sources show NOT STAGED (all 5 = 1,046 files). Also saves a full HDFS listing.

In [ ]:
printf '%-14s %8s %6s %8s %8s %8s  %s\n' source expected hdfs records mismatch repl!=2 result
all=PASS; total=0; exp_total=0
for r in "${SOURCES[@]}"; do
  IFS='|' read -r src s3 ver acq want mode <<< "$r"
  raw=$(raw_path "$src" "$ver" "$acq"); man=$(man_path "$src" "$ver" "$acq")
  want_src "$src" && exp_total=$((exp_total + want))
  if ! hdfs dfs -test -e "$raw" && ! want_src "$src"; then
    printf '%-14s %8s %6s %8s %8s %8s  %s\n' "$src" "$want" - - - - "NOT STAGED"; continue
  fi
  n=$(hdfs dfs -count "$raw" 2>/dev/null | awk '{print $2}'); n=${n:-0}
  read -r recs bad <<< "$(hdfs dfs -cat "$man/staging_record.csv" 2>/dev/null | awk -F, 'NR>1 {n++; if ($NF != "yes") b++} END {print n+0, b+0}')"
  rep=$(hdfs dfs -ls -R "$raw" 2>/dev/null | awk '$1 !~ /^d/ && NF >= 8 && $2 != 2' | wc -l)
  if [[ $n -eq $want && $recs -eq $want && $bad -eq 0 && $rep -eq 0 ]]; then res=PASS; else res=FAIL; all=FAIL; fi
  total=$((total + n))
  printf '%-14s %8s %6s %8s %8s %8s  %s\n' "$src" "$want" "$n" "$recs" "$bad" "$rep" "$res"
done
echo "total files in HDFS: $total (expected $exp_total)   overall: $all"
LISTING=$STAGE/hdfs_listing_$(date -u +%F).txt
hdfs dfs -ls -R /corpus/raw > "$LISTING" && echo "listing saved: $LISTING ($(wc -l < "$LISTING") lines)"
hdfs dfs -du -s -h /corpus/raw

## 6. (Optional) Save evidence to S3
HDFS and the primary node disappear with the cluster. Copy the manifests and listing out first.

In [ ]:
EV=$STAGE/evidence_staging_$(date -u +%Y%m%dT%H%M%SZ)
mkdir -p "$EV" && hdfs dfs -get /corpus/raw/_manifests "$EV/" && cp "$STAGE"/hdfs_listing_*.txt "$EV/"
aws s3 cp --recursive "$EV" "s3://$B/evidence/$(basename "$EV")/" --only-show-errors \
  && echo "uploaded: s3://$B/evidence/$(basename "$EV")/"

## 7. (Dev clusters only) Reset one source
**Deletes staged raw data.** Use only to retry a failed staging on a test cluster. Uncomment, set `SRC`, run.

In [ ]:
# SRC=gdelt
# IFS='|' read -r src s3 ver acq want mode <<< "$(src_row "$SRC")"
# hdfs dfs -rm -r -skipTrash "$(raw_path "$src" "$ver" "$acq")" "$(man_path "$src" "$ver" "$acq")"
# rm -rf "$STAGE/$src"